# 📘 Enterprise Textbook Pipeline Control Center

Interactive Jupyter dashboard for the existing automated textbook ingestion pipeline.

- Uses the tested `run_all_textbooks.py` runner.
- Runs pipeline commands with `workers/multimodal-ingestion/.venv/bin/python`.
- Keeps **Dry-run** enabled by default.
- Requires explicit confirmation before a real paid/remote execution.
- Shows registry books, adapters, state, progress, reports, paths, and logs.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import shlex
import subprocess
import sys
from collections import Counter
from pathlib import Path
from typing import Any

import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, clear_output, display


def locate_repo_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/home/ec2-user/SageMaker/Enterprise-Document-Intelligence"),
    ]

    for candidate in candidates:
        runner_path = (
            candidate
            / "workers/multimodal-ingestion/scripts/run_all_textbooks.py"
        )
        if runner_path.is_file():
            return candidate.resolve()

    raise FileNotFoundError(
        "Repository root not found. Upload this notebook to "
        "Enterprise-Document-Intelligence/notebooks/."
    )


REPO_ROOT = locate_repo_root()
PIPELINE_PYTHON = (
    REPO_ROOT
    / "workers/multimodal-ingestion/.venv/bin/python"
)
RUNNER_PATH = (
    REPO_ROOT
    / "workers/multimodal-ingestion/scripts/run_all_textbooks.py"
)
REGISTRY_PATH = (
    REPO_ROOT
    / "data/textbook-automation/ncert-i-x-book-registry.json"
)
STATE_PATH = (
    REPO_ROOT
    / "data/textbook-automation/ncert-i-x-auto-pipeline-state.json"
)
SCRIPTS_ROOT = (
    REPO_ROOT
    / "workers/multimodal-ingestion/scripts"
)
SRC_ROOT = (
    REPO_ROOT
    / "workers/multimodal-ingestion/src"
)

for import_root in (REPO_ROOT, SCRIPTS_ROOT, SRC_ROOT):
    value = str(import_root)
    if value not in sys.path:
        sys.path.insert(0, value)


def load_module(module_name: str, path: Path):
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not load module: {path}")

    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


runner = load_module(
    "textbook_pipeline_control_center_runner",
    RUNNER_PATH,
)

from processing_adapters import adapter_runtime_metadata


assert PIPELINE_PYTHON.is_file(), PIPELINE_PYTHON
assert RUNNER_PATH.is_file(), RUNNER_PATH
assert REGISTRY_PATH.is_file(), REGISTRY_PATH

print("Repository:", REPO_ROOT)
print("Pipeline Python:", PIPELINE_PYTHON)
print("Runner:", RUNNER_PATH)
print("Registry:", REGISTRY_PATH)
print("State:", STATE_PATH)
print()
print("Control Center setup: READY")


In [ ]:
def load_json_object(path: Path) -> dict[str, Any]:
    if not path.is_file():
        return {}

    value = json.loads(path.read_text(encoding="utf-8"))

    if not isinstance(value, dict):
        raise ValueError(f"Expected JSON object: {path}")

    return value


def load_registry_books() -> list[dict[str, Any]]:
    registry = load_json_object(REGISTRY_PATH)
    books = registry.get("books", [])

    if not isinstance(books, list):
        raise ValueError("Registry field 'books' must be a list.")

    return [
        dict(book)
        for book in books
        if isinstance(book, dict)
    ]


def state_record_map() -> dict[str, dict[str, Any]]:
    state = load_json_object(STATE_PATH)
    books = state.get("books", {})

    if not isinstance(books, dict):
        return {}

    return {
        str(book_id): dict(record)
        for book_id, record in books.items()
        if isinstance(record, dict)
    }


def build_registry_dataframe() -> pd.DataFrame:
    state_books = state_record_map()
    rows: list[dict[str, Any]] = []

    for book in load_registry_books():
        book_id = str(book.get("book_id", ""))
        adapter = adapter_runtime_metadata(book)
        state_record = state_books.get(book_id, {})

        persisted_state = state_record.get(
            "status",
            "NOT_STARTED",
        )

        relative_paths = (
            runner.paths_for_book(
                book_id,
                "v1",
            )
        )

        effective_paths = {
            name: (
                value
                if value.is_absolute()
                else REPO_ROOT / value
            )
            for name, value
            in relative_paths.items()
        }

        effective_stage = (
            runner.discover_current_stage(
                book_id,
                effective_paths,
            )
        )

        rows.append({
            "book_id": book_id,
            "grade": book.get("grade"),
            "title": book.get("title"),
            "subject": book.get("subject"),
            "language": book.get("language"),
            "adapter": adapter["adapter_key"],
            "script": adapter["expected_script"],
            "direction": adapter["reading_direction"],
            "persisted_state": persisted_state,
            "effective_stage": effective_stage,
            "stage_mismatch": (
                persisted_state
                != effective_stage
            ),
            # Keep the existing internal column
            # name for filters and progress.
            "state": effective_stage,
            "error": state_record.get("error"),
            "updated_at": state_record.get("updated_at"),
        })

    dataframe = pd.DataFrame(rows)

    if dataframe.empty:
        return dataframe

    return dataframe.sort_values(
        by=["grade", "subject", "book_id"],
        kind="stable",
    ).reset_index(drop=True)


def stage_progress(status: str) -> float:
    if status == "FAILED" or status not in runner.STAGES:
        return 0.0

    verified_index = runner.STAGES.index("VERIFIED")
    current_index = runner.STAGES.index(status)

    return round(
        min(current_index / verified_index, 1.0) * 100,
        1,
    )


def filtered_registry_dataframe() -> pd.DataFrame:
    dataframe = build_registry_dataframe()

    if dataframe.empty:
        return dataframe

    if grade_filter.value != "All":
        dataframe = dataframe[
            dataframe["grade"] == int(grade_filter.value)
        ]

    if adapter_filter.value != "All":
        dataframe = dataframe[
            dataframe["adapter"] == adapter_filter.value
        ]

    if status_filter.value != "All":
        dataframe = dataframe[
            dataframe["state"] == status_filter.value
        ]

    dataframe = dataframe.copy()
    dataframe["progress_percent"] = dataframe["state"].map(stage_progress)

    return dataframe.reset_index(drop=True)


def selected_book_ids() -> list[str]:
    available = set(
        filtered_registry_dataframe()["book_id"].tolist()
    )

    selected = [
        book_id
        for book_id in book_selector.value
        if book_id in available
    ]

    return selected or sorted(available)


def selected_grades(book_ids: list[str]) -> str:
    registry_by_id = {
        str(book.get("book_id")): book
        for book in load_registry_books()
    }

    grades = sorted({
        int(registry_by_id[book_id]["grade"])
        for book_id in book_ids
        if book_id in registry_by_id
    })

    if not grades:
        return (
            "1-10"
            if grade_filter.value == "All"
            else str(grade_filter.value)
        )

    return ",".join(str(grade) for grade in grades)


def build_pipeline_command() -> list[str]:
    book_ids = selected_book_ids()

    if not book_ids:
        raise ValueError("No books match the current filters.")

    bucket = bucket_input.value.strip()
    prefix = prefix_input.value.strip()

    if not bucket:
        raise ValueError("Bucket cannot be empty.")

    if not prefix:
        raise ValueError("Prefix cannot be empty.")

    command = [
        str(PIPELINE_PYTHON),
        str(RUNNER_PATH),
        "--bucket",
        bucket,
        "--prefix",
        prefix,
        "--grades",
        selected_grades(book_ids),
        "--registry",
        str(REGISTRY_PATH),
        "--state",
        str(STATE_PATH),
        "--max-retries",
        str(max_retries.value),
    ]

    for book_id in book_ids:
        command.extend(["--book-id", book_id])

    if through_stage_dropdown.value:
        command.extend([
            "--through-stage",
            through_stage_dropdown.value,
        ])

    if resume_checkbox.value:
        command.append("--resume")

    if dry_run_checkbox.value:
        command.append("--dry-run")

    return command


def command_as_shell() -> str:
    return " ".join(
        shlex.quote(part)
        for part in build_pipeline_command()
    )


def artifact_dataframe(book_id: str) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []

    for name, relative_path in sorted(
        runner.paths_for_book(book_id, "v1").items()
    ):
        absolute_path = REPO_ROOT / relative_path

        rows.append({
            "artifact": name,
            "relative_path": str(relative_path),
            "exists": absolute_path.exists(),
            "type": (
                "directory"
                if absolute_path.is_dir()
                else "file"
                if absolute_path.is_file()
                else "missing"
            ),
            "size_bytes": (
                absolute_path.stat().st_size
                if absolute_path.is_file()
                else None
            ),
        })

    return pd.DataFrame(rows)


def read_report_summary(path: Path) -> dict[str, Any]:
    payload = load_json_object(path)

    fields = (
        "status",
        "inspection_status",
        "metadata_enrichment_status",
        "processing_blocked_by_title_review",
        "document_count",
        "chapter_count",
        "canonical_page_count",
        "safe_fallback_title_count",
        "low_confidence_title_count",
        "generated_at",
        "updated_at",
    )

    return {
        field: payload.get(field)
        for field in fields
        if field in payload
    }


def tail_text(path: Path, line_count: int = 80) -> str:
    if not path.is_file():
        return f"Log file does not exist:\n{path}"

    lines = path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    return "\n".join(lines[-line_count:])

In [ ]:
registry_df = build_registry_dataframe()

available_grades = sorted({
    int(value)
    for value in registry_df["grade"].dropna().tolist()
})

available_adapters = sorted(
    registry_df["adapter"].dropna().unique().tolist()
)

available_statuses = sorted(
    registry_df["state"].dropna().unique().tolist()
)


grade_filter = widgets.Dropdown(
    options=["All", *[str(grade) for grade in available_grades]],
    value="All",
    description="Grade:",
)

adapter_filter = widgets.Dropdown(
    options=["All", *available_adapters],
    value="All",
    description="Adapter:",
)

status_filter = widgets.Dropdown(
    options=["All", *available_statuses],
    value="All",
    description="Status:",
)

book_selector = widgets.SelectMultiple(
    options=registry_df["book_id"].tolist(),
    value=(),
    rows=12,
    description="Books:",
    layout=widgets.Layout(width="98%"),
)

bucket_input = widgets.Text(
    value="edi-documents-ajam-2026",
    description="Bucket:",
    layout=widgets.Layout(width="95%"),
)

prefix_input = widgets.Text(
    value="NCERT I-X/",
    description="Prefix:",
    layout=widgets.Layout(width="95%"),
)

resume_checkbox = widgets.Checkbox(
    value=True,
    description="Resume from persisted state",
)

dry_run_checkbox = widgets.Checkbox(
    value=True,
    description="Dry-run only",
)

through_stage_dropdown = widgets.Dropdown(
    options=[
        ("Full pipeline", ""),
        (
            "Stop after CONFIG_GENERATED",
            "CONFIG_GENERATED",
        ),
    ],
    value="",
    description="Stop after:",
    layout=widgets.Layout(width="95%"),
)

paid_confirmation = widgets.Checkbox(
    value=False,
    description=(
        "I understand real execution may call "
        "AWS/BDA/Titan/OpenSearch/Surya"
    ),
    indent=False,
)

real_run_phrase = widgets.Text(
    value="",
    placeholder="Type RUN PAID PIPELINE",
    description="Confirm:",
    layout=widgets.Layout(width="95%"),
)

max_retries = widgets.BoundedIntText(
    value=2,
    min=0,
    max=10,
    description="Retries:",
)

refresh_button = widgets.Button(
    description="Refresh Dashboard",
    icon="refresh",
    button_style="info",
)

preview_button = widgets.Button(
    description="Preview Command",
    icon="search",
)

run_button = widgets.Button(
    description="Run Selected Pipeline",
    icon="play",
    button_style="danger",
)

details_book = widgets.Dropdown(
    options=registry_df["book_id"].tolist(),
    description="Inspect:",
    layout=widgets.Layout(width="95%"),
)

details_button = widgets.Button(
    description="Show Book Details",
    icon="folder-open",
)

dashboard_output = widgets.Output()
command_output = widgets.Output()
execution_output = widgets.Output()
details_output = widgets.Output()


def update_book_options(*_: object) -> None:
    dataframe = filtered_registry_dataframe()
    options = dataframe["book_id"].tolist()

    previous_selection = tuple(
        value
        for value in book_selector.value
        if value in options
    )

    book_selector.options = options
    book_selector.value = previous_selection
    details_book.options = options

    if options:
        details_book.value = options[0]


def render_dashboard(*_: object) -> None:
    update_book_options()
    dataframe = filtered_registry_dataframe()

    with dashboard_output:
        clear_output(wait=True)

        if dataframe.empty:
            print("No books match the current filters.")
            return

        status_counts = Counter(dataframe["state"].tolist())

        display(
            HTML(
                "<h3>Pipeline Dashboard</h3>"
                f"<b>Matching books:</b> {len(dataframe)}<br>"
                f"<b>Status counts:</b> "
                f"{dict(sorted(status_counts.items()))}"
            )
        )

        display(
            dataframe[
                [
                    "grade",
                    "book_id",
                    "title",
                    "subject",
                    "language",
                    "adapter",
                    "persisted_state",
                    "effective_stage",
                    "stage_mismatch",
                    "progress_percent",
                    "error",
                ]
            ]
        )


def preview_command(*_: object) -> None:
    with command_output:
        clear_output(wait=True)

        try:
            print("Selected book count:", len(selected_book_ids()))
            print(
                "Execution mode:",
                "DRY-RUN"
                if dry_run_checkbox.value
                else "REAL EXECUTION",
            )
            print(
                "Stop after stage:",
                through_stage_dropdown.value
                or "FULL_PIPELINE",
            )
            print()
            print()
            print(command_as_shell())
            print()

            if dry_run_checkbox.value:
                print("Remote processing calls expected: 0")
            else:
                print(
                    "WARNING: Real execution may call "
                    "AWS/BDA/Titan/OpenSearch/Surya."
                )

        except Exception as error:
            print(f"Cannot build command: {error}")


def run_pipeline(*_: object) -> None:
    with execution_output:
        clear_output(wait=True)

        if not dry_run_checkbox.value:
            if not paid_confirmation.value:
                print(
                    "REAL EXECUTION BLOCKED: "
                    "enable paid-processing confirmation."
                )
                return

            if (
                real_run_phrase.value.strip()
                != "RUN PAID PIPELINE"
            ):
                print(
                    'REAL EXECUTION BLOCKED: type exactly '
                    '"RUN PAID PIPELINE".'
                )
                return

        try:
            command = build_pipeline_command()
        except Exception as error:
            print(f"Cannot start pipeline: {error}")
            return

        print("Starting command:")
        print(
            " ".join(
                shlex.quote(part)
                for part in command
            )
        )
        print("=" * 100)

        process = subprocess.Popen(
            command,
            cwd=REPO_ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        assert process.stdout is not None

        for line in process.stdout:
            print(line, end="")

        return_code = process.wait()

        print("=" * 100)
        print("Process return code:", return_code)
        render_dashboard()


def show_book_details(*_: object) -> None:
    with details_output:
        clear_output(wait=True)

        book_id = details_book.value

        if not book_id:
            print("No book selected.")
            return

        print("=" * 100)
        print("BOOK STATE")
        print("=" * 100)
        print(
            json.dumps(
                state_record_map().get(book_id, {}),
                indent=2,
                ensure_ascii=False,
                default=str,
            )
        )

        print()
        print("=" * 100)
        print("ARTIFACT PATHS")
        print("=" * 100)
        display(artifact_dataframe(book_id))

        paths = runner.paths_for_book(book_id, "v1")

        report_paths = {
            "inspection": REPO_ROOT / paths["inspection"],
            "generation_report": (
                REPO_ROOT / paths["generation_report"]
            ),
            "extraction_report": (
                REPO_ROOT / paths["extraction_report"]
            ),
            "merge_report": REPO_ROOT / paths["merge_report"],
            "bda_bridge_report": (
                REPO_ROOT / paths["bda_bridge_report"]
            ),
            "ocr_plan": REPO_ROOT / paths["ocr_plan"],
            "ocr_fallback_report": (
                REPO_ROOT / paths["ocr_fallback_report"]
            ),
        }

        summaries = []

        for report_name, report_path in report_paths.items():
            summary = read_report_summary(report_path)

            if summary:
                summaries.append({
                    "report": report_name,
                    "path": str(
                        report_path.relative_to(REPO_ROOT)
                    ),
                    **summary,
                })

        print()
        print("=" * 100)
        print("REPORT SUMMARIES")
        print("=" * 100)

        if summaries:
            display(pd.DataFrame(summaries))
        else:
            print("No readable reports available.")

        print()
        print("=" * 100)
        print("LATEST LOG LINES")
        print("=" * 100)
        print(tail_text(REPO_ROOT / paths["log"]))


for control in (
    grade_filter,
    adapter_filter,
    status_filter,
):
    control.observe(
        render_dashboard,
        names="value",
    )

refresh_button.on_click(render_dashboard)
preview_button.on_click(preview_command)
run_button.on_click(run_pipeline)
details_button.on_click(show_book_details)


display(
    HTML(
        """
        <style>
        .edi-panel {
            border: 1px solid #d9d9d9;
            border-radius: 10px;
            padding: 14px;
            margin: 8px 0;
            background: #fafafa;
        }
        </style>
        <h2>📚 Textbook Automation Dashboard</h2>
        <p>
        Filter books, preview the command, and execute only
        after reviewing the safety controls.
        </p>
        """
    )
)

display(
    widgets.VBox([
        widgets.HTML(
            "<div class='edi-panel'><b>1. Filter books</b></div>"
        ),
        widgets.HBox([
            grade_filter,
            adapter_filter,
            status_filter,
        ]),
        book_selector,
        refresh_button,
        dashboard_output,
        widgets.HTML(
            "<div class='edi-panel'><b>2. Pipeline settings</b></div>"
        ),
        bucket_input,
        prefix_input,
        widgets.HBox([
            resume_checkbox,
            dry_run_checkbox,
            max_retries,
        ]),
        through_stage_dropdown,
        paid_confirmation,
        real_run_phrase,
        widgets.HBox([
            preview_button,
            run_button,
        ]),
        command_output,
        execution_output,
        widgets.HTML(
            "<div class='edi-panel'><b>3. Inspect state, reports, paths and logs</b></div>"
        ),
        details_book,
        details_button,
        details_output,
    ])
)

render_dashboard()

## Recommended operating sequence

1. Keep **Dry-run only** enabled first.
2. Filter by grade, adapter, or status.
3. Select one or more books. Empty selection means all currently filtered books.
4. Click **Preview Command**.
5. Run the dry-run and review the planned operations.
6. For a real run, disable dry-run, enable the paid-processing checkbox, and type `RUN PAID PIPELINE`.
7. Use **Show Book Details** to inspect persisted state, report summaries, artifact paths, and logs.

A dry-run should make no BDA, Titan, Bedrock, OpenSearch, S3-write, or Surya processing calls.
